In [1]:
import os
import joblib
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

MODEL_A_PATH = Path("../../saved_model_A_final") # Твоя обученная Модель А
ONNX_A_PATH = Path("../../onnx_models/model_A_quantized") # Куда сохраняем
ENCODER_PATH = Path("../../label_encoder_model_a.joblib")
TEST_DATA_PATH = Path("../data/processed/gold_test_holdout.csv")

ONNX_A_PATH.mkdir(parents=True, exist_ok=True)

print(f"Экспорт модели А в ONNX...")
ort_model = ORTModelForSequenceClassification.from_pretrained(
    MODEL_A_PATH, 
    export=True
)

print(f"Запуск квантизации (динамический int8)...")
quantizer = ORTQuantizer.from_pretrained(ort_model)

qconfig = AutoQuantizationConfig.avx2(
    is_static=False,    # динамическая квантизация
    per_channel=False 
)

quantizer.quantize(
    save_dir=ONNX_A_PATH,
    quantization_config=qconfig,
)

# проверка веса (safetensors vs onnx)
orig_size = os.path.getsize(MODEL_A_PATH / "model.safetensors") / (1024 * 1024)
quant_size = os.path.getsize(ONNX_A_PATH / "model_quantized.onnx") / (1024 * 1024)

print(f"\nКвантизация завершена!")
print(f"Размер оригинала: {orig_size:.2f} MB")
print(f"Размер после сжатия: {quant_size:.2f} MB")
print(f"Сжатие в {orig_size / quant_size:.2f} раз")

# ВАЛИДАЦИЯ КАЧЕСТВА (Hold-out тест)
print(f"\n🧪 Запуск валидации на Hold-out наборе (100 строк)...")

test_df = pd.read_csv(TEST_DATA_PATH)
label2id = joblib.load(ENCODER_PATH)
all_labels = list(label2id.keys())

# загрузка квантизированной модели и токенизатора
quantized_model = ORTModelForSequenceClassification.from_pretrained(ONNX_A_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_A_PATH)

y_true = test_df['label'].map(label2id).values
predictions = []

for text in tqdm(test_df['text'].tolist()):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    outputs = quantized_model(**inputs)
    
    logits = outputs.logits
    if hasattr(logits, 'detach'):
        logits = logits.detach().numpy()
        
    predictions.append(np.argmax(logits))

new_f1 = f1_score(y_true, predictions, average='macro')
original_f1 = 0.82  # результат из предыдущего шага

print("\n" + "="*40)
print(f"Итоговый отчет для диплома:")
print(f"F1-score ОРИГИНАЛЬНОЙ Модели А: {original_f1:.4f}")
print(f"F1-score КВАНТИЗИРОВАННОЙ Модели А: {new_f1:.4f}")

f1_drop = (original_f1 - new_f1) / original_f1 * 100
print(f"Падение качества: {f1_drop:.2f}%")

if f1_drop > 2:
    print("Падение качества более 2%.")
else:
    print("Потери минимальны!")
print("="*40)

print("\nДетальный отчет квантизированной модели:")
print(classification_report(y_true, predictions, target_names=all_labels))

W0521 16:24:27.189000 26492 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Экспорт модели А в ONNX...
Запуск квантизации (динамический int8)...


Could not find any ONNX files with standard file name model.onnx, files found: [WindowsPath('model_quantized.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.



Квантизация завершена!
Размер оригинала: 44.99 MB
Размер после сжатия: 11.41 MB
Сжатие в 3.94 раз

🧪 Запуск валидации на Hold-out наборе (100 строк)...


100%|█████████████| 100/100 [00:00<00:00, 228.82it/s]



Итоговый отчет для диплома:
F1-score ОРИГИНАЛЬНОЙ Модели А: 0.8200
F1-score КВАНТИЗИРОВАННОЙ Модели А: 0.8196
Падение качества: 0.05%
Потери минимальны!

Детальный отчет квантизированной модели:


ValueError: Number of classes, 29, does not match size of target_names, 31. Try specifying the labels parameter